NN_EVALUATED_PUFs
================
This notebook generates the datasets needed to evaluate several PUF types, including iPUF, XOR-PUF, and PCL-PUF.
It contains the generation scripts and supporting code for neural-network-based PUF analysis.

It includes:
- feature construction for arbiter-PUF-style challenges,
- REAP-NVM and PC-LPUF simulation code,
- obfuscation logic for position/value encoding,
- dataset generation routines for interpose, XOR, and PC-LPUF variants.

You can change the PUF configuration settings to recreate the data used in the paper under different settings.

In [ ]:
"""

"""

import numpy as np
from scipy.io import savemat  # Optional MATLAB export for saved results
from pypuf.io import random_inputs
import math
from pypuf.simulation import ArbiterPUF


# -----------------------------------------------------------------------------
# 1. Challenge feature construction for arbiter-PUF-style models
# -----------------------------------------------------------------------------
def arbiter_features_binary(challenges):
    """Convert challenge bits in {-1, +1} into arbiter-PUF parity features."""
    # Reverse cumulative product matches the standard arbiter-PUF feature map.
    phi = np.flip(np.cumprod(np.flip(challenges, axis=1), axis=1), axis=1)
    # Add a bias term so the model can learn an intercept-like offset.
    return np.hstack([phi, np.ones((phi.shape[0], 1))])


# -----------------------------------------------------------------------------
# 2. REAP-NVM PUF simulation helpers
def ReapNVM(num_bits, seed, sigma_proc = 0.05):
    rng = np.random.default_rng(seed)

    chal_length = num_bits
    n_levels = 4

    # ---- Fixed nominal resistance levels ----
    R_levels_nom = np.array([10e3, 75e3, 125e3, 275e3])

    # ---- Convert to log domain ----
    log_R_levels_nom = np.log10(R_levels_nom)

    # ---- Allocate output arrays ----
    tR = np.zeros((2, n_levels, chal_length))

    # ---- Generate per-cell values ----
    for row in range(2):
        for stage in range(chal_length):

            # Add Gaussian process variation in log domain
            log_levels = log_R_levels_nom + sigma_proc * rng.standard_normal(n_levels)

            # Convert back to linear domain
            levels = 10 ** log_levels

            # RC delay mapping
            tR[row, :, stage] = levels * 250e-12

    # ---- Deterministic switching delay ----
    tSW = np.full((2, 2, chal_length), 372.0 / 1e12)

    return 4.0 * tR, 4.0 * tSW

def ReapNVM_evaluate(PUF, challenge, position, value, chunk_size=10_000_0):
    """
    Evaluate the REAP-NVM PUF for many challenges in chunks.
    This implements the timing-based delay comparison used by the lower layer.
    """
    chal = (challenge + 1) / 2.0
    tR4, tSW4 = PUF
    chalpos = position.astype(int).flatten()
    chalval = value.astype(int).flatten()

    num_challenges, challenge_length = chal.shape
    responses = np.zeros(num_challenges, dtype=np.int8)

    # Process the challenge set in chunks for memory efficiency.
    for start in range(0, num_challenges, chunk_size):
        end = min(start + chunk_size, num_challenges)
        N = end - start

        chal_chunk = chal[start:end, :]
        pos_chunk = chalpos[start:end]
        val_chunk = chalval[start:end]

        # Base timing vectors for the two possible transition paths.
        tv1_chunk = np.tile(tR4[0, 0, :], (N, 1))
        tv2_chunk = np.tile(tR4[1, 0, :], (N, 1))

        # Apply per-challenge stage modification for the selected position/value.
        tv1_chunk[np.arange(N), pos_chunk] = tR4[0, val_chunk, pos_chunk]
        tv2_chunk[np.arange(N), pos_chunk] = tR4[1, val_chunk, pos_chunk]

        # Compute the cumulative XOR path indicator for each challenge.
        c_chunk = np.bitwise_xor.accumulate(chal_chunk.astype(np.uint8), axis=1)

        # Use the stage-wise timing values to compute the final delay comparison.
        t1_arr = np.where(
            c_chunk == 0,
            tv1_chunk + tSW4[0, 0, :],
            tv2_chunk + tSW4[1, 0, :]
        )

        t2_arr = np.where(
            c_chunk == 0,
            tv2_chunk + tSW4[0, 1, :],
            tv1_chunk + tSW4[1, 1, :]
        )

        # Sum the stage delays and compare the two paths.
        t1_sum = np.sum(t1_arr, axis=1)
        t2_sum = np.sum(t2_arr, axis=1)

        responses[start:end] = (t1_sum > t2_sum).astype(np.int8)

    return responses


# -----------------------------------------------------------------------------
# 3. PC-LPUF construction using upper arbiter PUFs and lower REAP-NVM PUFs
# -----------------------------------------------------------------------------
def NOVEL_PCLPUF(num_bits, xor_puf_up, xor_puf_down, seed=2):
    """Build a PC-LPUF instance with upper-layer Arbiter PUFs and lower-layer REAP-NVM PUFs."""
    lower_puf = []
    for i in range(xor_puf_down):
        lower_puf.append(ReapNVM(num_bits, i + seed))

    upper_pufs = []
    number_of_upper_pufs = 3 + math.log2(num_bits)
    for i in range(xor_puf_up):
        upper_pufs.append(ArbiterPUF(n=128, seed=seed + i, noisiness=0))
    return [lower_puf, upper_pufs]


# -----------------------------------------------------------------------------
# 4. Obfuscation helpers for position/value encoding
# -----------------------------------------------------------------------------
def sliding_window_xor(bits, window_bits):
    """Apply a sliding-window XOR transform to a binary vector using the upper-layer response bits."""
    P = bits.shape[0]
    U = window_bits.shape[0]
    window_bits = window_bits.astype(bits.dtype)

    # Slide the window across the bits and XOR each segment with the corresponding window.
    for start in range(0, P, U):
        end = min(start + U, P)
        w = end - start
        bits[start:end] ^= window_bits[:w]

    return bits


def xor_obfuscate_position_value(position, value, upper_resp):
    """
    Obfuscate the position/value inputs using the upper-layer response bits.
    This mimics the PC-LPUF's hidden mapping layer.
    """
    N = position.shape[0]
    U = upper_resp.shape[0]

    pos_out = np.zeros(N, dtype=np.uint32)
    val_out = np.zeros(N, dtype=np.uint32)

    for n in range(N):
        # Use the upper-layer response vector as a sliding window for the obfuscation step.
        window_bits = upper_resp[:, n]

        # Convert decimal values into binary vectors before applying XOR obfuscation.
        pos_bits = dec_to_bin_vec(position[n], 7)
        val_bits = dec_to_bin_vec(value[n], 2)

        # XOR-transform the encoded position and value bits.
        pos_bits = sliding_window_xor(pos_bits, window_bits)
        val_bits = sliding_window_xor(val_bits, window_bits)

        # Convert the modified bits back to decimal values.
        pos_out[n] = bin_vec_to_dec(pos_bits)
        val_out[n] = bin_vec_to_dec(val_bits)

    return pos_out, val_out


def dec_to_bin_vec(x, bitlen):
    """Convert an unsigned integer to a binary vector of fixed length."""
    return np.array([(x >> i) & 1 for i in range(bitlen)][::-1], dtype=np.uint8)


def bin_vec_to_dec(bits):
    """Convert a binary vector back to an unsigned integer."""
    out = 0
    for b in bits:
        out = (out << 1) | int(b)
    return out


# -----------------------------------------------------------------------------
# 5. Full PC-LPUF evaluation
# -----------------------------------------------------------------------------
def NOVEL_PCLPUF_evaluate(PUF, challenge, position, value):
    """Evaluate a full PC-LPUF instance for the given challenge, position, and value inputs."""
    reap_pufs = PUF[0]
    upper_arbiter_pufs = PUF[1]
    upper_puf_responses = []

    # Evaluate the upper-layer Arbiter PUFs on the challenge input.
    for puf in upper_arbiter_pufs:
        upper_puf_responses.append(puf.eval(challenge[:, :128]))

    upper_puf_responses = np.array(upper_puf_responses)
    binary_array = (upper_puf_responses + 1) // 2  # Convert to {0, 1}
    N, L = challenge.shape

    # Apply the obfuscation mapping to the position/value data.
    last6, first3 = xor_obfuscate_position_value(position, value, binary_array)

    responses = []
    for puf in reap_pufs:
        responses.append(ReapNVM_evaluate(puf, challenge, last6, first3))

    print(np.array(responses).shape)
    return [np.bitwise_xor.reduce(np.array(responses), axis=0), responses]


In [ ]:
###############
# Interpose-PUF data generation
# This cell creates a dataset for an Interpose PUF using randomly generated challenges.
# The resulting file stores the challenge features and the corresponding binary responses.

import numpy as np
import pypuf.simulation
from pypuf.simulation import InterposePUF
from pypuf.io import random_inputs

# -----------------------------------------------------------------------------
# Configuration for the Interpose-PUF dataset
# -----------------------------------------------------------------------------
num_pufs = 1
num_challenges = 1_000_000
num_bits = 128

Up_PUF = 3
Down_PUF = 4
challenges = random_inputs(n=num_bits, N=num_challenges, seed=42)

N, L = challenges.shape

# Compute challenge features once and reuse them for all generated PUF instances.
X = arbiter_features_binary(challenges)

for puf_id in range(num_pufs):
    # Instantiate one Interpose PUF with a distinct seed.
    puf = InterposePUF(
        n=num_bits,
        k_up=Up_PUF,
        k_down=Down_PUF,
        seed=1 + puf_id,
        noisiness=0
    )

    # Evaluate the PUF on all generated challenges.
    responses = puf.eval(challenges)
    y = np.array(responses)
    y = (y + 1) / 2.0  # Convert from {-1, +1} to {0, 1}

    # Save the dataset as a compressed NumPy archive.
    filename = f"puf_data_interpose_puf_up{Up_PUF}_down{Down_PUF}_{puf_id}.npz"
    np.savez_compressed(filename, X=X, y=y)

    print(f"Saved {filename}")


In [ ]:
# XOR-PUF data generation
# This cell generates a dataset for an XOR Arbiter PUF and stores the
# challenge features together with the corresponding response labels.

import numpy as np
import pypuf.simulation
from pypuf.simulation import InterposePUF
from pypuf.io import random_inputs
from pypuf.simulation import XORArbiterPUF


# -----------------------------------------------------------------------------
# Configuration for the XOR-PUF dataset
# -----------------------------------------------------------------------------
num_pufs = 1
num_challenges = 10_000_000
num_bits = 128
k = 8  # Number of XORed arbiter PUFs

# Fixed challenges for a fair comparison across experiments.
challenges = random_inputs(n=num_bits, N=num_challenges, seed=42)

N, L = challenges.shape

# Compute challenge features once so the same mapping is used for all samples.
X = arbiter_features_binary(challenges)

for puf_id in range(num_pufs):
    # Create a new XOR Arbiter PUF instance with a different seed.
    puf = XORArbiterPUF(n=num_bits, k=k, seed=1 + puf_id)

    # Evaluate the PUF on the generated challenge set.
    responses = puf.eval(challenges)
    y = np.array(responses)
    y = (y + 1) / 2.0  # Convert from {-1, +1} to {0, 1}

    # Save the generated dataset to disk.
    filename = f"puf_data_xor{k}_puf_attack_{puf_id}.npz"
    np.savez_compressed(filename, X=X, y=y)

    print(f"Saved {filename}")


In [ ]:
# PCL-PUF data generation
# This cell generates a synthetic PCL-PUF dataset by sampling challenges,
# position/value obfuscation inputs, and the resulting responses from the model.

import numpy as np


# -----------------------------------------------------------------------------
# Configuration for the PCL-PUF dataset
# -----------------------------------------------------------------------------
def dec_to_bin_vec(x, bitlen):
    """Convert an integer to a binary vector of fixed length."""
    return np.array([(x >> i) & 1 for i in range(bitlen)][::-1], dtype=np.uint8)

num_pufs = 1
num_challenges = 10_000_000
num_bits = 128
k_up = 1
k_down = 1

# Fixed challenges for a fair comparison across experiments.
challenge = random_inputs(n=num_bits, N=num_challenges, seed=42)
value = np.random.randint(0, 4, size=(num_challenges,))
position = np.random.randint(0, num_bits, size=(num_challenges,))

# Encode the position and value inputs as binary vectors.
pos_len = 7
val_len = 2

pos_bin = np.array(
    [list(np.binary_repr(x, width=pos_len)) for x in position],
    dtype=np.uint8
)

val_bin = np.array(
    [list(np.binary_repr(x, width=val_len)) for x in value],
    dtype=np.uint8
)

# Challenge features are computed once and reused for every generated sample.
X_chal = arbiter_features_binary(challenge)

for puf_id in range(num_pufs):
    print(f"\nGenerating NOVEL-IPUF dataset for PUF {puf_id}")

    # Build the PCL-PUF instance with the chosen upper/down-layer sizes.
    PUF = NOVEL_PCLPUF(
        num_bits,
        xor_puf_up=k_up,
        xor_puf_down=k_down,
        seed=puf_id * 10 + 100
    )

    # Evaluate the PUF on the generated challenges with the obfuscated inputs.
    [responses, each_puf] = NOVEL_PCLPUF_evaluate(
        PUF,
        challenge,
        position,
        value
    )

    y = responses

    # Assemble the final input matrix by combining challenge features with
    # the encoded position and value information.
    inp = np.zeros((num_challenges, num_bits + 10), dtype=np.int8)
    inp[:, :num_bits + 1] = X_chal
    inp[:, num_bits + 1 : num_bits + 8] = pos_bin
    inp[:, num_bits + 8 : num_bits + 10] = val_bin

    # Save the generated dataset as a compressed NumPy archive.
    filename = f"puf_data_novel_PCLPUF_up{k_up}_down{k_down}_puf{puf_id}.npz"
    np.savez_compressed(filename, X=inp, y=y)

    print(f"Saved {filename}")


In [ ]:
import numpy as np

# -----------------------------------------------------------------------------
# REAP-NVM dataset generation
# This cell generates a dataset for a single REAP-NVM PUF instance and saves
# the challenge features together with the encoded position/value inputs.
# -----------------------------------------------------------------------------

num_pufs = 1
num_challenges = 10_000_000
num_bits = 128

# Fixed randomness for challenges
challenge = random_inputs(n=num_bits, N=num_challenges, seed=42)
value = np.random.randint(0, 4, size=(num_challenges,))
position = np.random.randint(0, num_bits, size=(num_challenges,))

# Encode position and value as one-hot vectors.
pos_len = 7
val_len = 2

pos_bin = np.eye(num_bits, dtype=int)[position]  # shape: (num_challenges, num_bits)
num_values = 4
val_bin = np.eye(num_values, dtype=int)[value]  # shape: (num_challenges, 4)

# Challenge features
X_chal = arbiter_features_binary(challenge)

for puf_id in range(num_pufs):
    print(f"\nGenerating dataset for PUF {puf_id}")

    # Generate one REAP-NVM PUF instance and evaluate it on all challenges.
    reap = ReapNVM(num_bits, seed=10 + puf_id * 100)
    y = ReapNVM_evaluate(
        reap,
        challenge,
        position,
        value
    )

    start = num_bits + 1

    inp = np.zeros((num_challenges, start + pos_bin.shape[1] + val_bin.shape[1]), dtype=np.int8)

    inp[:, :start] = X_chal
    inp[:, start:start + pos_bin.shape[1]] = pos_bin
    inp[:, start + pos_bin.shape[1]:] = val_bin

    filename = f"puf_data_reap_puf_{puf_id}.npz"
    np.savez_compressed(filename, X=inp, y=y)

    print(f"Saved {filename}")
